# Нейросетевая классификация патологий ритма сердца

Этот ноутбук обучает **1D-CNN модель** для классификации сердечных сокращений на 5 классов по стандарту AAMI:

- **N** - Normal (нормальные)
- **S** - Supraventricular ectopic (наджелудочковые экстрасистолы)
- **V** - Ventricular ectopic (желудочковые экстрасистолы)
- **F** - Fusion (сливные комплексы)
- **Q** - Unknown (артефакты/неизвестные)

## Архитектура

Используется **1D Convolutional Neural Network** с residual-блоками ? стандартная архитектура для классификации ЭКГ. Свёртки эффективно извлекают локальные морфологические признаки QRS-комплекса, а residual-связи стабилизируют обучение.

## Входные данные

Файл `mitbih_balanced.npz`, содержащий три варианта обучающей выборки:
1. Исходная несбалансированная + взвешенный loss
2. После SMOTE
3. После аугментаций сигнала

## 1. Импорт библиотек

In [2]:
# !pip install torch numpy matplotlib scikit-learn seaborn tqdm

In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.metrics import (
    confusion_matrix, classification_report,
    f1_score, precision_score, recall_score, accuracy_score
)

# Фиксация seed для воспроизводимости
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Устройство: cpu


## 2. Загрузка сбалансированных данных

In [4]:
data = np.load('./models/mitbih_balanced.npz', allow_pickle=True)

print('Доступные ключи:')
for key in data.files:
    arr = data[key]
    if arr.ndim == 0:
        print(f'  {key:20s} | scalar = {arr}')
    else:
        print(f'  {key:20s} | shape = {arr.shape}')

CLASS_NAMES = list(data['class_names'])
N_CLASSES = len(CLASS_NAMES)
print(f'\nКлассы: {CLASS_NAMES}')

Доступные ключи:
  X_train              | shape = (51001, 250)
  y_train              | shape = (51001,)
  X_train_smote        | shape = (137539, 250)
  y_train_smote        | shape = (137539,)
  X_train_aug          | shape = (106975, 250)
  y_train_aug          | shape = (106975,)
  X_test               | shape = (49691, 250)
  y_test               | shape = (49691,)
  class_weights        | shape = (5,)
  sample_weights       | shape = (51001,)
  class_names          | shape = (5,)
  fs                   | scalar = 360

Классы: ['N', 'S', 'V', 'F', 'Q']


### Выбор стратегии балансировки

Можно переключаться между тремя вариантами:
- `'aug'` ? аугментации сигнала **(рекомендуется для ЭКГ)**
- `'smote'` ? SMOTE-интерполяция
- `'weighted'` ? несбалансированные данные + взвешенный loss

In [5]:
STRATEGY = 'aug'  # 'aug' | 'smote' | 'weighted'

if STRATEGY == 'aug':
    X_train = data['X_train_aug']
    y_train = data['y_train_aug']
    use_weighted_loss = False
elif STRATEGY == 'smote':
    X_train = data['X_train_smote']
    y_train = data['y_train_smote']
    use_weighted_loss = False
elif STRATEGY == 'weighted':
    X_train = data['X_train']
    y_train = data['y_train']
    use_weighted_loss = True
else:
    raise ValueError(f'Unknown strategy: {STRATEGY}')

X_test = data['X_test']
y_test = data['y_test']
class_weights = data['class_weights']

print(f'Стратегия: {STRATEGY}')
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Взвешенный loss: {use_weighted_loss}')

Стратегия: aug
Train: (106975, 250), Test: (49691, 250)
Взвешенный loss: False


### Валидационная выборка

Выделяем 15% обучающей выборки для отслеживания переобучения. Используем стратификацию для сохранения баланса классов.

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.15,
    stratify=y_train,
    random_state=SEED
)

print(f'Train: {X_train.shape}')
print(f'Val:   {X_val.shape}')
print(f'Test:  {X_test.shape}')

Train: (90928, 250)
Val:   (16047, 250)
Test:  (49691, 250)


## 3. PyTorch Dataset и DataLoader

Создаём кастомный `Dataset` для удобной работы с PyTorch.
Сигнал ЭКГ имеет shape `(250,)`, но для 1D-свёртки нужен формат `(C, T)` = `(1, 250)`, где `C` - число каналов.

In [7]:
class ECGDataset(Dataset):
    """Dataset для ЭКГ-сегментов."""
    
    def __init__(self, X, y):
        # Добавляем размерность канала: (N, 250) ? (N, 1, 250)
        self.X = torch.from_numpy(X).float().unsqueeze(1)
        self.y = torch.from_numpy(y).long()
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


BATCH_SIZE = 256

train_ds = ECGDataset(X_train, y_train)
val_ds = ECGDataset(X_val, y_val)
test_ds = ECGDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

# Проверка формы батча
x_sample, y_sample = next(iter(train_loader))
print(f'\nФорма батча X: {x_sample.shape} (batch, channels, time)')
print(f'Форма батча y: {y_sample.shape}')

Train batches: 356
Val batches:   63
Test batches:  195

Форма батча X: torch.Size([256, 1, 250]) (batch, channels, time)
Форма батча y: torch.Size([256])


## 4. Архитектура нейросети: 1D ResNet

Используем сверточную сеть с **residual-блоками**:

### Структура
1. **Stem** - начальный свёрточный слой для извлечения базовых признаков
2. **3 residual-блока** с возрастающим числом каналов (32 - 64 - 128)
3. **Global Average Pooling** - агрегация по временной оси
4. **Dropout + Linear** - классификатор

### Почему residual-блоки?
Residual-связи $y = F(x) + x$ позволяют обучать более глубокие сети без проблем с затухающими градиентами. Для ЭКГ это особенно важно, так как одновременно нужны:
- **Локальные признаки** (форма QRS-комплекса)
- **Глобальные признаки** (общая форма цикла)

In [8]:
class ResidualBlock1D(nn.Module):
    """
    Residual-блок для 1D-сигнала.
    
    Структура:
      x ──┬──> Conv ? BN ? ReLU ? Conv ? BN ──┐
          │                                    + ? ReLU
          └──────────[1x1 Conv (если нужно)]──┘
    """
    
    def __init__(self, in_channels, out_channels, kernel_size=5, stride=1, dropout=0.2):
        super().__init__()
        padding = kernel_size // 2
        
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size,
                                stride=stride, padding=padding, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size,
                                stride=1, padding=padding, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        
        self.dropout = nn.Dropout(dropout)
        
        # Shortcut для согласования размерностей
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()
    
    def forward(self, x):
        identity = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        out = out + identity
        return F.relu(out)


class ECGResNet(nn.Module):
    """
    1D ResNet для классификации ЭКГ-сегментов.
    
    Параметры:
    ----------
    n_classes : int
        Количество классов
    base_channels : int
        Базовое количество каналов
    dropout : float
        Вероятность dropout
    """
    
    def __init__(self, n_classes=5, base_channels=32, dropout=0.3):
        super().__init__()
        
        # Stem: начальное извлечение признаков
        self.stem = nn.Sequential(
            nn.Conv1d(1, base_channels, kernel_size=7, stride=1, padding=3, bias=False),
            nn.BatchNorm1d(base_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),  # 250 ? 125
        )
        
        # Residual-блоки с увеличением каналов и уменьшением длины
        self.block1 = ResidualBlock1D(base_channels, base_channels, stride=1, dropout=0.2)
        self.block2 = ResidualBlock1D(base_channels, base_channels*2, stride=2, dropout=0.2)  # 125 ? 63
        self.block3 = ResidualBlock1D(base_channels*2, base_channels*4, stride=2, dropout=0.3)  # 63 ? 32
        
        # Global Average Pooling агрегирует временную ось
        self.gap = nn.AdaptiveAvgPool1d(1)
        
        # Классификатор
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(base_channels*4, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )
    
    def forward(self, x):
        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x


# Инициализация и проверка
model = ECGResNet(n_classes=N_CLASSES, base_channels=32, dropout=0.3).to(DEVICE)

# Подсчёт параметров
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Количество обучаемых параметров: {n_params:,}')

# Проверка forward pass
with torch.no_grad():
    test_input = torch.randn(2, 1, 250).to(DEVICE)
    test_output = model(test_input)
    print(f'Вход:  {test_input.shape}')
    print(f'Выход: {test_output.shape}')

Количество обучаемых параметров: 184,229
Вход:  torch.Size([2, 1, 250])
Выход: torch.Size([2, 5])


## 5. Настройка обучения

### Функция потерь
Используем **CrossEntropyLoss** (с весами классов, если выбрана стратегия `'weighted'`).

### Оптимизатор
**Adam** с начальным `lr=1e-3` ? хорошо работает для большинства задач.

### Scheduler
**ReduceLROnPlateau** уменьшает learning rate при выходе на плато по валидационному loss.

In [9]:
# Функция потерь
if use_weighted_loss:
    weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)
    print(f'Используется взвешенный CrossEntropyLoss с весами: {class_weights}')
else:
    criterion = nn.CrossEntropyLoss()
    print('Используется обычный CrossEntropyLoss')

# Оптимизатор
LR = 1e-3
WEIGHT_DECAY = 1e-4
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Scheduler
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print(f'\nОптимизатор: Adam, lr={LR}, weight_decay={WEIGHT_DECAY}')

Используется обычный CrossEntropyLoss

Оптимизатор: Adam, lr=0.001, weight_decay=0.0001


## 6. Функции обучения и валидации

In [10]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Одна эпоха обучения."""
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * x.size(0)
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
        all_labels.append(y.cpu().numpy())
    
    avg_loss = total_loss / len(loader.dataset)
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro', zero_division=0)
    
    return avg_loss, acc, f1


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Оценка модели на выборке."""
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        
        total_loss += loss.item() * x.size(0)
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
        all_labels.append(y.cpu().numpy())
    
    avg_loss = total_loss / len(loader.dataset)
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro', zero_division=0)
    
    return avg_loss, acc, f1, preds, labels

## 7. Обучение модели

Используем **Early Stopping** - останавливаем обучение, если макро-F1 на валидации не растёт в течение `patience` эпох.


При дисбалансе классов accuracy завышена. Macro-F1 усредняет F1 по классам с равным весом, поэтому миноритарные классы важны так же, как мажоритарные.

## Цикл обучения

In [11]:
N_EPOCHS = 30
PATIENCE = 7  # Early Stopping: остановка, если F1 не растёт N эпох
BEST_MODEL_PATH = 'best_ecg_model.pth'

# История метрик для визуализации
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [],  'val_acc': [],
    'train_f1': [],   'val_f1': [],
    'lr': []
}

best_val_f1 = 0.0
epochs_no_improve = 0

print(f'Начало обучения на {DEVICE}')
print(f'Стратегия: {STRATEGY} | Эпох: {N_EPOCHS} | Patience: {PATIENCE}')
print('=' * 80)

for epoch in range(1, N_EPOCHS + 1):
    # === Обучение ===
    train_loss, train_acc, train_f1 = train_epoch(
        model, train_loader, criterion, optimizer, DEVICE
    )
    
    # === Валидация ===
    val_loss, val_acc, val_f1, _, _ = evaluate(
        model, val_loader, criterion, DEVICE
    )
    
    # === Scheduler ===
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # === Сохранение истории ===
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)
    history['lr'].append(current_lr)
    
    # === Логирование ===
    marker = ''
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        epochs_no_improve = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': val_f1,
            'val_acc': val_acc,
        }, BEST_MODEL_PATH)
        marker = ' ★ (saved)'
    else:
        epochs_no_improve += 1
    
    print(f'Эпоха {epoch:3d}/{N_EPOCHS} | '
          f'Train loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | '
          f'Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f} | '
          f'lr={current_lr:.1e}{marker}')
    
    # === Early Stopping ===
    if epochs_no_improve >= PATIENCE:
        print(f'\n⚠ Early stopping: F1 не улучшается {PATIENCE} эпох')
        break

print('=' * 80)
print(f'✓ Обучение завершено. Лучший val F1 = {best_val_f1:.4f}')
print(f'✓ Модель сохранена в {BEST_MODEL_PATH}')

Начало обучения на cpu
Стратегия: aug | Эпох: 30 | Patience: 7
Эпоха   1/30 | Train loss=0.3413 acc=0.8885 f1=0.8731 | Val loss=0.1906 acc=0.9483 f1=0.9426 | lr=1.0e-03 ★ (saved)
Эпоха   2/30 | Train loss=0.1371 acc=0.9586 f1=0.9530 | Val loss=0.0752 acc=0.9771 f1=0.9747 | lr=1.0e-03 ★ (saved)
Эпоха   3/30 | Train loss=0.1073 acc=0.9666 f1=0.9623 | Val loss=0.0639 acc=0.9792 f1=0.9768 | lr=1.0e-03 ★ (saved)
Эпоха   4/30 | Train loss=0.0895 acc=0.9725 f1=0.9689 | Val loss=0.0493 acc=0.9846 f1=0.9830 | lr=1.0e-03 ★ (saved)
Эпоха   5/30 | Train loss=0.0783 acc=0.9759 f1=0.9729 | Val loss=0.0516 acc=0.9837 f1=0.9820 | lr=1.0e-03
Эпоха   6/30 | Train loss=0.0666 acc=0.9793 f1=0.9769 | Val loss=0.0434 acc=0.9864 f1=0.9851 | lr=1.0e-03 ★ (saved)
Эпоха   7/30 | Train loss=0.0627 acc=0.9805 f1=0.9781 | Val loss=0.0381 acc=0.9870 f1=0.9857 | lr=1.0e-03 ★ (saved)
Эпоха   8/30 | Train loss=0.0542 acc=0.9829 f1=0.9808 | Val loss=0.0396 acc=0.9864 f1=0.9852 | lr=1.0e-03
Эпоха   9/30 | Train loss=0.0